In [1]:
# Imports & Configuration
from pathlib import Path

import duckdb
import polars as pl
from great_tables import GT

# Project paths
ROOT = Path.cwd()
while not (ROOT / "raw_data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW_DIR = ROOT / "raw_data"

# DuckDB connection
con = duckdb.connect()

In [2]:
# ============================================================
# Functions
# ============================================================

def show_sql(query: str, max_rows: int = 100):
    """Execute DuckDB SQL and display results using Great Tables."""
    result = con.sql(query).limit(max_rows)

    df = pl.DataFrame(
        result.fetchall(),
        schema=result.columns,
        orient="row"
    )

    display(GT(df))

In [3]:
# 01 | SKU Referential Integrity

show_sql(f"""
    WITH orders AS (
        SELECT 'CZ' AS shop, sku
        FROM read_csv('{(RAW_DIR / "raw_orders_cz.csv").as_posix()}')

        UNION ALL

        SELECT 'SK', sku
        FROM read_csv('{(RAW_DIR / "raw_orders_sk.csv").as_posix()}')

        UNION ALL

        SELECT 'HU', sku
        FROM read_csv('{(RAW_DIR / "raw_orders_hu.csv").as_posix()}')
    ),
    products AS (
        SELECT sku
        FROM read_csv('{(RAW_DIR / "raw_products.csv").as_posix()}')
    )
    SELECT
        o.shop,
        COUNT(DISTINCT o.sku) AS ordered_skus,
        COUNT(DISTINCT o.sku) FILTER (
            WHERE p.sku IS NULL
        ) AS missing_skus,
        COUNT(*) FILTER (
            WHERE p.sku IS NULL
        ) AS affected_order_lines
    FROM orders o
    LEFT JOIN products p ON o.sku = p.sku
    GROUP BY o.shop
    ORDER BY o.shop
""")

shop,ordered_skus,missing_skus,affected_order_lines
CZ,32,4,180
HU,32,4,34
SK,32,4,83


In [4]:
# 02 | FX Coverage

show_sql(f"""
    WITH orders AS (
        SELECT
            currency,
            CAST(
                CAST(created_at AS TIMESTAMPTZ)
                AT TIME ZONE 'Europe/Prague'
                AS DATE
            ) AS order_date
        FROM read_csv('{(RAW_DIR / "raw_orders_sk.csv").as_posix()}')

        UNION ALL

        SELECT
            currency,
            CAST(
                CAST(created_at AS TIMESTAMPTZ)
                AT TIME ZONE 'Europe/Prague'
                AS DATE
            )
        FROM read_csv('{(RAW_DIR / "raw_orders_hu.csv").as_posix()}')
    ),
    fx AS (
        SELECT
            currency,
            CAST(date AS DATE) AS rate_date
        FROM read_csv('{(RAW_DIR / "raw_fx_rates.csv").as_posix()}')
    )
    SELECT
        o.currency,
        COUNT(*) AS order_lines,
        COUNT(*) FILTER (
            WHERE NOT EXISTS (
                SELECT 1
                FROM fx
                WHERE fx.currency = o.currency
                  AND fx.rate_date <= o.order_date
            )
        ) AS missing_historical_fx
    FROM orders o
    GROUP BY o.currency
    ORDER BY o.currency
""")

currency,order_lines,missing_historical_fx
EUR,5439,0
HUF,3101,0


In [5]:
# 03 | Missing Product References

show_sql(f"""
    WITH orders AS (
        SELECT 'CZ' AS shop, sku
        FROM read_csv('{(RAW_DIR / "raw_orders_cz.csv").as_posix()}')
        UNION ALL
        SELECT 'SK', sku
        FROM read_csv('{(RAW_DIR / "raw_orders_sk.csv").as_posix()}')
        UNION ALL
        SELECT 'HU', sku
        FROM read_csv('{(RAW_DIR / "raw_orders_hu.csv").as_posix()}')
    )
    SELECT
        o.sku,
        COUNT(*) AS affected_lines,
        STRING_AGG(DISTINCT o.shop, ', ' ORDER BY o.shop) AS shops
    FROM orders o
    LEFT JOIN read_csv('{(RAW_DIR / "raw_products.csv").as_posix()}') p
        ON o.sku = p.sku
    WHERE p.sku IS NULL
    GROUP BY o.sku
    ORDER BY affected_lines DESC
""")

sku,affected_lines,shops
FB-PAN-24-LTD,113,"CZ, HU, SK"
FB-WOK-30-BLK,79,"CZ, HU, SK"
FB-KNF-UTIL,63,"CZ, HU, SK"
FB-POT-28,42,"CZ, HU, SK"


| Area | Finding | Recommendation |
|---|---|---|
| SKU integrity | 4 SKUs missing from the product catalog, affecting 297 raw order lines across CZ, SK and HU | Preserve unmatched records and flag missing product references |
| Product costs | Missing SKUs have no available unit costs | Do not assume or invent costs; report the limitation |
| FX coverage | All 8,540 raw SK/HU order lines have historical FX coverage | Use the latest available historical rate |
| Currency conversion | EUR and HUF require conversion to CZK | Apply FX conversion in the business layer |

**Staging:** Preserve unmatched SKUs and source values.

**Business logic:** Handle missing product costs explicitly and document their impact on margin calculations.